h(t) = A * h(t-1) + B * x(t)     # update the hidden state
y(t) = C * h(t)                  # produce an output

h is the hidden state ("running summary" of the sequence so far)
x is the input @ each step
y is the output

A, B, C are matrices that control the behavior


In [1]:
import torch
import torch.nn as nn

test code to ensure algorithm works

In [2]:
input_dim = 4   # size of x(t)
hidden_dim = 8  # size of h(t)
output_dim = 4  # size of y(t)

A = torch.randn(hidden_dim, hidden_dim)  # hidden -> hidden
B = torch.randn(hidden_dim, input_dim)   # input -> hidden
C = torch.randn(output_dim, hidden_dim)  # hidden -> output

def ssm_step(x, h_prev):
    h = A @ h_prev + B @ x  # h(t) = A * h(t-1) + B * x(t)
    y = C @ h               # y(t) = C * h(t)
    return h, y

seq_len = 10
x_seq = torch.randn(seq_len, input_dim)
h = torch.zeros(hidden_dim)
outputs = []

for t in range(seq_len):
    h, y = ssm_step(x_seq[t], h)
    outputs.append(y)

In [4]:
class SelectiveSSM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.A = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.linear_B = nn.Linear(input_dim, hidden_dim)
        self.Linear_C = nn.Linear(input_dim, output_dim)
        self.hidden_dim = hidden_dim
        
    def forward(self, x_seq):
        batch_size, seq_len, _ = x_seq.shape
        h = torch.zeros(batch_size, self.hidden_dim)
        outputs = []
        
        for t in range(seq_len):
            x_t = x_seq[:, t, :]
            B_t = self.linear_B(x_t)   # generated from input
            C_t = self.linear_C(x_t)   # generated from input
            
            h = self.A @ h.unsqueeze(-1)
            h = h.squeeze(-1) + B_t
            y = C_t * h
            y = y.sum(dim = -1, keepdim = True)
            
            outputs.append(y)
            
        return torch.cat(outputs, dim = -1)